In [ ]:
import requests
import time
import json
import pandas as pd
from typing import List, Dict, Any

# Load JSON into a list of dicts
df = pd.read_json("/content/unique_locations.json")
locations: List[Dict[str, Any]] = df.to_dict(orient="records")

NOMINATIM_URL = "https://nominatim.openstreetmap.org/reverse"
HEADERS = {"User-Agent": "notebook-geocoder/1.0 (your_email@example.com)"}  # replace with your contact info

def pick_place_name(address: Dict[str, Any]) -> str:
    """Choose a human-friendly place name from Nominatim's address dict."""
    primary = ("city", "town", "village", "hamlet", "municipality")
    secondary = ("county", "region", "state_district", "state", "province")
    country = address.get("country")
    parts = []
    for key in primary:
        if key in address:
            parts.append(address[key])
            break
    if not parts:
        for key in secondary:
            if key in address:
                parts.append(address[key])
                break
    if country:
        if not parts or parts[-1] != country:
            parts.append(country)
    if not parts:
        return address.get("display_name", "Unknown location")
    return ", ".join(parts)

def reverse_geocode(lat: float, lon: float, zoom: int = 10, sleep_s: float = 1.0) -> Dict[str, Any]:
    """Call Nominatim reverse endpoint and return parsed info."""
    params = {"format": "jsonv2", "lat": lat, "lon": lon, "zoom": zoom, "addressdetails": 1}
    try:
        r = requests.get(NOMINATIM_URL, params=params, headers=HEADERS, timeout=15)
        r.raise_for_status()
        data = r.json()
        address = data.get("address", {})
        place_name = pick_place_name(address if isinstance(address, dict) else {})
        return {
            "lat": lat,
            "lon": lon,
            "place_name": place_name,
            "display_name": data.get("display_name"),
            "address": address
        }
    except requests.RequestException as e:
        return {"lat": lat, "lon": lon, "place_name": "ERROR", "error": str(e)}
    finally:
        time.sleep(sleep_s)  # respect rate limits

# Main loop
output: List[Dict[str, Any]] = []
print(f"Reverse-geocoding {len(locations)} locations (1 req/sec)")

for i, loc in enumerate(locations, start=1):
    lat = float(loc["lat"])
    lon = float(loc["lon"])
    result = reverse_geocode(lat, lon, zoom=10, sleep_s=1.0)
    output.append(result)

    if i % 10 == 0 or i == len(locations):
        print(f"Processed {i}/{len(locations)}")

# Save results
out_path = "locations_with_names.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Saved {len(output)} items to {out_path}")

from pprint import pprint
pprint(output[:10])
